In [21]:
!pip install playwright pandas beautifulsoup4 requests tqdm nest_asyncio
!playwright install

In [22]:
import requests
import pandas as pd
import nest_asyncio

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

from tqdm import tqdm
from playwright.async_api import async_playwright

In [23]:
nest_asyncio.apply()

In [24]:
def crawl_site(start_url, max_pages=50):

    visited = set()
    queue = [start_url]
    pages = []

    domain = urlparse(start_url).netloc

    while queue and len(pages) < max_pages:

        url = queue.pop(0)

        if url in visited:
            continue

        visited.add(url)
        pages.append(url)

        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, "html.parser")

            for link in soup.find_all("a", href=True):

                new_url = urljoin(url, link["href"])

                if urlparse(new_url).netloc == domain:
                    if new_url not in visited:
                        queue.append(new_url)

        except:
            pass

    return pages

In [25]:
async def scan_page(page, url):

    await page.goto(url)

    await page.add_script_tag(
        url="https://cdnjs.cloudflare.com/ajax/libs/axe-core/4.8.2/axe.min.js"
    )

    results = await page.evaluate(
        """async () => {
            return await axe.run();
        }"""
    )

    return results["violations"]

In [26]:
def parse_violations(page_url, violations):

    rows = []

    for v in violations:

        for node in v["nodes"]:

            rows.append({
                "page": page_url,
                "violation": v["id"],
                "severity": v["impact"],
                "description": v["description"],
                "content": node["html"],
                "recommended_fix": v["help"]
            })

    return rows

In [31]:
start_url = "www.example.com"

pages = crawl_site(start_url, max_pages=20)

print("Pages discovered:", len(pages))
pages

Pages discovered: 1


['www.example.com']

In [28]:
async def run_scan(pages):

    all_rows = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for url in tqdm(pages):

            try:

                violations = await scan_page(page, url)

                rows = parse_violations(url, violations)

                all_rows.extend(rows)

            except Exception as e:

                print("Error scanning:", url)

        await browser.close()

    return all_rows

In [29]:
all_rows = await run_scan(pages)

100%|███████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 199.94it/s]

Error scanning: honolulupd.org


In [30]:
df = pd.DataFrame(all_rows)

df.head()

""
